In [1]:
# Cell 2 – Imports
import pandas as pd
from sqlalchemy import create_engine, text
import os

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Cell 3 – Load cleaned CSV & create SQLite database
CSV_PATH = "../data/cleaned_superstore.csv"
DB_PATH  = "../data/superstore.db"

df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape}")
print(df.head(3))

Shape: (6764, 21)
   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
1       3  CA-2016-138688  2016-06-12  2016-06-16    Second Class    DV-13045   
2       5  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
2   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   South  FUR-BO-10001798        Furniture    Bookcases   
1       90036    West  OFF-LA-10000240  Office Supplies       Labels   
2       33311   South  OFF-ST-10000760  Office Supplies      Storage   

                                        Product Name    Sales  Quantity  \
0           

In [3]:
# Cell 4 – Write DataFrame to SQLite
engine = create_engine(f'sqlite:///{DB_PATH}')

df.to_sql('superstore', con=engine, if_exists='replace', index=False)
print(f'Table superstore written to {DB_PATH}')

Table superstore written to ../data/superstore.db


In [4]:
# Cell 5 – Reusable query utility
def run_query(sql, label=''):
    """Execute SQL and return a DataFrame."""
    with engine.connect() as conn:
        result = pd.read_sql(text(sql), conn)
    if label:
        print(f'\n{"="*55}\n  {label}\n{"="*55}')
        display(result)
    return result

# Verify
run_query('SELECT COUNT(*) AS total_rows FROM superstore', 'Row Count')


  Row Count


,total_rows
0,6764


,total_rows
0,6764


---
## Day 7-8 – SQL Fundamentals

In [5]:
# Cell 6 – Overall KPIs
run_query("""
SELECT
    COUNT(DISTINCT "Order ID")   AS unique_orders,
    COUNT(DISTINCT "Customer ID") AS unique_customers,
    ROUND(SUM(Sales), 2)         AS total_sales,
    ROUND(SUM(Profit), 2)        AS total_profit
FROM superstore
""", 'Overall KPIs')


  Overall KPIs


,unique_orders,unique_customers,total_sales,total_profit
0,4050,787,459978.39,78934.94


,unique_orders,unique_customers,total_sales,total_profit
0,4050,787,459978.39,78934.94


In [6]:
# Cell 7 – Sales by Region
run_query("""
SELECT Region,
       ROUND(SUM(Sales), 2)  AS total_sales,
       ROUND(SUM(Profit), 2) AS total_profit
FROM superstore
GROUP BY Region
ORDER BY total_sales DESC
""", 'Sales by Region')


  Sales by Region


,Region,total_sales,total_profit
0,West,179078.70,30061.75
1,East,114325.16,21070.62
2,Central,93531.17,14837.44
3,South,73043.35,12965.13


,Region,total_sales,total_profit
0,West,179078.70,30061.75
1,East,114325.16,21070.62
2,Central,93531.17,14837.44
3,South,73043.35,12965.13


In [7]:
# Cell 8 – Sales by Category
run_query("""
SELECT Category,
       ROUND(SUM(Sales), 2)  AS total_sales,
       ROUND(SUM(Profit), 2) AS total_profit,
       COUNT(*)               AS order_count
FROM superstore
GROUP BY Category
ORDER BY total_sales DESC
""", 'Sales by Category')


  Sales by Category


,Category,total_sales,total_profit,order_count
0,Office Supplies,197919.73,49169.71,4512
1,Furniture,139295.23,13126.49,1159
2,Technology,122763.43,16638.74,1093


,Category,total_sales,total_profit,order_count
0,Office Supplies,197919.73,49169.71,4512
1,Furniture,139295.23,13126.49,1159
2,Technology,122763.43,16638.74,1093


In [8]:
# Cell 9 – Top 10 loss-making orders
run_query("""
SELECT "Order ID", "Product Name",
       ROUND(Sales, 2) AS sales,
       ROUND(Profit, 2) AS profit
FROM superstore
WHERE Profit < 0
ORDER BY Profit ASC
LIMIT 10
""", 'Top 10 Loss-Making Orders')


  Top 10 Loss-Making Orders


,Order ID,Product Name,sales,profit
0,US-2015-156867,Logitech K350 2.4Ghz Wireless Keyboard,238.90,-26.88
1,CA-2015-120901,Fellowes Stor/Drawer Steel Plus Storage Drawers,152.69,-26.72
2,CA-2017-151750,Office Star - Contemporary Task Swivel Chair,310.74,-26.64
3,CA-2015-116484,"Belkin 19"" Center-Weighted Shelf, Gray",141.55,-26.54
4,CA-2017-121293,Fellowes Bankers Box Staxonsteel Drawer File/S...,259.92,-25.99
5,CA-2015-157812,Carina Double Wide Media Storage Towers in Nat...,129.57,-25.91
6,CA-2014-123295,Office Star - Ergonomically Designed Knee Chair,259.14,-25.91
7,CA-2017-160122,Global Highback Leather Tilter in Burgundy,127.39,-25.48
8,CA-2017-104850,"Global Wood Trimmed Manager's Task Chair, Khaki",291.14,-25.47
9,CA-2015-118955,Office Star - Contemporary Swivel Chair with P...,197.37,-25.38


,Order ID,Product Name,sales,profit
0,US-2015-156867,Logitech K350 2.4Ghz Wireless Keyboard,238.90,-26.88
1,CA-2015-120901,Fellowes Stor/Drawer Steel Plus Storage Drawers,152.69,-26.72
2,CA-2017-151750,Office Star - Contemporary Task Swivel Chair,310.74,-26.64
3,CA-2015-116484,"Belkin 19"" Center-Weighted Shelf, Gray",141.55,-26.54
4,CA-2017-121293,Fellowes Bankers Box Staxonsteel Drawer File/S...,259.92,-25.99
5,CA-2015-157812,Carina Double Wide Media Storage Towers in Nat...,129.57,-25.91
6,CA-2014-123295,Office Star - Ergonomically Designed Knee Chair,259.14,-25.91
7,CA-2017-160122,Global Highback Leather Tilter in Burgundy,127.39,-25.48
8,CA-2017-104850,"Global Wood Trimmed Manager's Task Chair, Khaki",291.14,-25.47
9,CA-2015-118955,Office Star - Contemporary Swivel Chair with P...,197.37,-25.38


In [9]:
# Cell 9 – Top 10 loss-making orders
run_query("""
SELECT "Order ID", "Product Name",
       ROUND(Sales, 2) AS sales,
       ROUND(Profit, 2) AS profit
FROM superstore
WHERE Profit < 0
ORDER BY Profit ASC
LIMIT 10
""", 'Top 10 Loss-Making Orders')


  Top 10 Loss-Making Orders


,Order ID,Product Name,sales,profit
0,US-2015-156867,Logitech K350 2.4Ghz Wireless Keyboard,238.90,-26.88
1,CA-2015-120901,Fellowes Stor/Drawer Steel Plus Storage Drawers,152.69,-26.72
2,CA-2017-151750,Office Star - Contemporary Task Swivel Chair,310.74,-26.64
3,CA-2015-116484,"Belkin 19"" Center-Weighted Shelf, Gray",141.55,-26.54
4,CA-2017-121293,Fellowes Bankers Box Staxonsteel Drawer File/S...,259.92,-25.99
5,CA-2015-157812,Carina Double Wide Media Storage Towers in Nat...,129.57,-25.91
6,CA-2014-123295,Office Star - Ergonomically Designed Knee Chair,259.14,-25.91
7,CA-2017-160122,Global Highback Leather Tilter in Burgundy,127.39,-25.48
8,CA-2017-104850,"Global Wood Trimmed Manager's Task Chair, Khaki",291.14,-25.47
9,CA-2015-118955,Office Star - Contemporary Swivel Chair with P...,197.37,-25.38


,Order ID,Product Name,sales,profit
0,US-2015-156867,Logitech K350 2.4Ghz Wireless Keyboard,238.90,-26.88
1,CA-2015-120901,Fellowes Stor/Drawer Steel Plus Storage Drawers,152.69,-26.72
2,CA-2017-151750,Office Star - Contemporary Task Swivel Chair,310.74,-26.64
3,CA-2015-116484,"Belkin 19"" Center-Weighted Shelf, Gray",141.55,-26.54
4,CA-2017-121293,Fellowes Bankers Box Staxonsteel Drawer File/S...,259.92,-25.99
5,CA-2015-157812,Carina Double Wide Media Storage Towers in Nat...,129.57,-25.91
6,CA-2014-123295,Office Star - Ergonomically Designed Knee Chair,259.14,-25.91
7,CA-2017-160122,Global Highback Leather Tilter in Burgundy,127.39,-25.48
8,CA-2017-104850,"Global Wood Trimmed Manager's Task Chair, Khaki",291.14,-25.47
9,CA-2015-118955,Office Star - Contemporary Swivel Chair with P...,197.37,-25.38


---
## Day 9-10 – Advanced SQL

In [10]:
# Cell 10 – CTE: Top 5 products by sales
run_query("""
WITH product_sales AS (
    SELECT "Product Name",
           ROUND(SUM(Sales), 2)  AS total_sales,
           ROUND(SUM(Profit), 2) AS total_profit
    FROM superstore
    GROUP BY "Product Name"
)
SELECT * FROM product_sales
ORDER BY total_sales DESC
LIMIT 5
""", 'CTE – Top 5 Products')


  CTE – Top 5 Products


,Product Name,total_sales,total_profit
0,"Situations Contoured Folding Chairs, 4/Set",2321.05,31.94
1,DMI Arturo Collection Mission-style Design Woo...,2204.31,81.53
2,Fellowes Officeware Wire Shelving,2155.92,43.12
3,Space Solutions HD Industrial Steel Shelving.,2069.46,62.08
4,Easy-staple paper,2047.09,875.59


,Product Name,total_sales,total_profit
0,"Situations Contoured Folding Chairs, 4/Set",2321.05,31.94
1,DMI Arturo Collection Mission-style Design Woo...,2204.31,81.53
2,Fellowes Officeware Wire Shelving,2155.92,43.12
3,Space Solutions HD Industrial Steel Shelving.,2069.46,62.08
4,Easy-staple paper,2047.09,875.59


In [11]:
# Cell 11 – CTE: Monthly revenue trend
run_query("""
WITH monthly AS (
    SELECT SUBSTR("Order Date", 7, 4) || '-' || SUBSTR("Order Date", 1, 2) AS yr_month,
           ROUND(SUM(Sales), 2) AS monthly_sales
    FROM superstore
    GROUP BY yr_month
)
SELECT * FROM monthly
ORDER BY yr_month
""", 'CTE – Monthly Revenue Trend')


  CTE – Monthly Revenue Trend


,yr_month,monthly_sales
0,0-01-20,1690.63
1,0-02-20,1592.24
2,0-03-20,1280.71
3,0-04-20,1177.77
4,0-05-20,1738.19
...,...,...
302,9-26-20,2488.88
303,9-27-20,371.34
304,9-28-20,1191.72
305,9-29-20,1958.88


,yr_month,monthly_sales
0,0-01-20,1690.63
1,0-02-20,1592.24
2,0-03-20,1280.71
3,0-04-20,1177.77
4,0-05-20,1738.19
...,...,...
302,9-26-20,2488.88
303,9-27-20,371.34
304,9-28-20,1191.72
305,9-29-20,1958.88


In [12]:
# Cell 12 – Window Function: RANK customers by spend
run_query("""
SELECT "Customer Name", Segment,
       ROUND(SUM(Sales), 2) AS total_spend,
       RANK() OVER (ORDER BY SUM(Sales) DESC) AS spend_rank
FROM superstore
GROUP BY "Customer Name", Segment
LIMIT 15
""", 'Window – Customer Spend Ranking')


  Window – Customer Spend Ranking


,Customer Name,Segment,total_spend,spend_rank
0,Joel Eaton,Consumer,2244.71,1
1,Arthur Gainer,Consumer,2211.25,2
2,Paul Prost,Home Office,2163.62,3
3,Sanjit Chand,Consumer,2038.72,4
4,Adrian Barton,Consumer,1968.97,5
5,Sally Hughsby,Corporate,1775.10,6
6,William Brown,Consumer,1764.45,7
7,Jay Kimmel,Consumer,1752.48,8
8,Kunst Miller,Consumer,1738.63,9
9,Bill Donatelli,Consumer,1737.15,10


,Customer Name,Segment,total_spend,spend_rank
0,Joel Eaton,Consumer,2244.71,1
1,Arthur Gainer,Consumer,2211.25,2
2,Paul Prost,Home Office,2163.62,3
3,Sanjit Chand,Consumer,2038.72,4
4,Adrian Barton,Consumer,1968.97,5
5,Sally Hughsby,Corporate,1775.10,6
6,William Brown,Consumer,1764.45,7
7,Jay Kimmel,Consumer,1752.48,8
8,Kunst Miller,Consumer,1738.63,9
9,Bill Donatelli,Consumer,1737.15,10


In [13]:
# Cell 13 – Window Function: LAG month-on-month change
run_query("""
WITH monthly AS (
    SELECT SUBSTR("Order Date", 7, 4) || '-' || SUBSTR("Order Date", 1, 2) AS yr_month,
           ROUND(SUM(Sales), 2) AS revenue
    FROM superstore
    GROUP BY yr_month
),
with_lag AS (
    SELECT yr_month, revenue,
           LAG(revenue) OVER (ORDER BY yr_month) AS prev_revenue
    FROM monthly
)
SELECT yr_month, revenue, prev_revenue,
       ROUND(revenue - prev_revenue, 2) AS change_abs,
       ROUND((revenue - prev_revenue) * 100.0 / prev_revenue, 1) AS pct_change
FROM with_lag
WHERE prev_revenue IS NOT NULL
ORDER BY yr_month
""", 'Window – Month-on-Month Change (LAG)')


  Window – Month-on-Month Change (LAG)


,yr_month,revenue,prev_revenue,change_abs,pct_change
0,0-02-20,1592.24,1690.63,-98.39,-5.8
1,0-03-20,1280.71,1592.24,-311.53,-19.6
2,0-04-20,1177.77,1280.71,-102.94,-8.0
3,0-05-20,1738.19,1177.77,560.42,47.6
4,0-06-20,1013.33,1738.19,-724.86,-41.7
...,...,...,...,...,...
301,9-26-20,2488.88,2393.93,94.95,4.0
302,9-27-20,371.34,2488.88,-2117.54,-85.1
303,9-28-20,1191.72,371.34,820.38,220.9
304,9-29-20,1958.88,1191.72,767.16,64.4


,yr_month,revenue,prev_revenue,change_abs,pct_change
0,0-02-20,1592.24,1690.63,-98.39,-5.8
1,0-03-20,1280.71,1592.24,-311.53,-19.6
2,0-04-20,1177.77,1280.71,-102.94,-8.0
3,0-05-20,1738.19,1177.77,560.42,47.6
4,0-06-20,1013.33,1738.19,-724.86,-41.7
...,...,...,...,...,...
301,9-26-20,2488.88,2393.93,94.95,4.0
302,9-27-20,371.34,2488.88,-2117.54,-85.1
303,9-28-20,1191.72,371.34,820.38,220.9
304,9-29-20,1958.88,1191.72,767.16,64.4


In [14]:
# Cell 14 – Create Views
with engine.connect() as conn:
    conn.execute(text("""
        CREATE VIEW IF NOT EXISTS v_subcategory_profit AS
        SELECT Category, "Sub-Category",
               ROUND(SUM(Sales), 2)   AS total_sales,
               ROUND(SUM(Profit), 2)  AS total_profit,
               ROUND(SUM(Profit)*100.0/SUM(Sales), 2) AS margin_pct
        FROM superstore
        GROUP BY Category, "Sub-Category"
    """))
    conn.execute(text("""
        CREATE VIEW IF NOT EXISTS v_customer_ltv AS
        SELECT "Customer ID", "Customer Name", Segment, Region,
               ROUND(SUM(Sales), 2)        AS lifetime_value,
               ROUND(SUM(Profit), 2)       AS lifetime_profit,
               COUNT(DISTINCT "Order ID")  AS total_orders
        FROM superstore
        GROUP BY "Customer ID", "Customer Name", Segment, Region
    """))
    conn.commit()
print('Views created successfully!')

run_query('SELECT * FROM v_subcategory_profit ORDER BY margin_pct DESC',
          'View – Sub-Category Profit Margins')

Views created successfully!

  View – Sub-Category Profit Margins


,Category,Sub-Category,total_sales,total_profit,margin_pct
0,Office Supplies,Labels,7401.54,3248.15,43.88
1,Office Supplies,Paper,38603.20,16293.75,42.21
2,Office Supplies,Envelopes,7809.44,3251.58,41.64
3,Office Supplies,Binders,25625.91,9876.88,38.54
4,Office Supplies,Fasteners,2730.65,877.50,32.14
5,Office Supplies,Art,21861.14,5213.61,23.85
6,Furniture,Furnishings,44295.56,9488.59,21.42
7,Office Supplies,Appliances,22643.82,4593.58,20.29
8,Technology,Accessories,47210.47,8269.97,17.52
9,Technology,Machines,2490.01,395.38,15.88


,Category,Sub-Category,total_sales,total_profit,margin_pct
0,Office Supplies,Labels,7401.54,3248.15,43.88
1,Office Supplies,Paper,38603.20,16293.75,42.21
2,Office Supplies,Envelopes,7809.44,3251.58,41.64
3,Office Supplies,Binders,25625.91,9876.88,38.54
4,Office Supplies,Fasteners,2730.65,877.50,32.14
5,Office Supplies,Art,21861.14,5213.61,23.85
6,Furniture,Furnishings,44295.56,9488.59,21.42
7,Office Supplies,Appliances,22643.82,4593.58,20.29
8,Technology,Accessories,47210.47,8269.97,17.52
9,Technology,Machines,2490.01,395.38,15.88


---
## 10 Business Questions

In [15]:
# Cell 15 – Run all 10 business questions and store results
biz_questions = {
    'Q1 – Top 5 Products by Sales': """
        SELECT "Product Name", ROUND(SUM(Sales),2) AS total_sales
        FROM superstore GROUP BY "Product Name"
        ORDER BY total_sales DESC LIMIT 5""",

    'Q2 – Monthly Revenue Trend': """
        SELECT SUBSTR("Order Date",7,4)||'-'||SUBSTR("Order Date",1,2) AS yr_month,
               ROUND(SUM(Sales),2) AS monthly_sales
        FROM superstore GROUP BY yr_month ORDER BY yr_month""",

    'Q3 – Customer Segmentation': """
        SELECT "Customer Name", Segment,
               ROUND(SUM(Sales),2) AS total_spend,
               CASE WHEN SUM(Sales)>=5000 THEN 'High Value'
                    WHEN SUM(Sales)>=1000 THEN 'Mid Value'
                    ELSE 'Low Value' END AS tier
        FROM superstore GROUP BY "Customer Name", Segment
        ORDER BY total_spend DESC LIMIT 20""",

    'Q4 – Most Profitable Ship Mode': """
        SELECT "Ship Mode", ROUND(SUM(Sales),2) AS total_sales,
               ROUND(SUM(Profit),2) AS total_profit, COUNT(*) AS orders
        FROM superstore GROUP BY "Ship Mode"
        ORDER BY total_profit DESC""",

    'Q5 – Top 5 Loss-Making Sub-Categories': """
        SELECT "Sub-Category", ROUND(SUM(Profit),2) AS total_profit
        FROM superstore GROUP BY "Sub-Category"
        ORDER BY total_profit ASC LIMIT 5""",

    'Q6 – Top 10 States by Revenue': """
        SELECT State, ROUND(SUM(Sales),2) AS total_sales,
               ROUND(SUM(Profit),2) AS total_profit
        FROM superstore GROUP BY State
        ORDER BY total_sales DESC LIMIT 10""",

    'Q7 – Discount vs Profit Impact': """
        SELECT CASE WHEN Discount=0    THEN 'No Discount'
                    WHEN Discount<=0.1 THEN '1-10%'
                    WHEN Discount<=0.2 THEN '11-20%'
                    WHEN Discount<=0.3 THEN '21-30%'
                    ELSE 'Above 30%' END AS discount_band,
               COUNT(*) AS orders,
               ROUND(AVG(Profit),2) AS avg_profit
        FROM superstore GROUP BY discount_band""",

    'Q8 – Avg Order Value by Segment': """
        SELECT Segment, ROUND(AVG(order_val),2) AS avg_order_value
        FROM (SELECT Segment, "Order ID", SUM(Sales) AS order_val
              FROM superstore GROUP BY Segment, "Order ID")
        GROUP BY Segment ORDER BY avg_order_value DESC""",

    'Q9 – Year-over-Year Sales': """
        SELECT SUBSTR("Order Date",7,4) AS year,
               ROUND(SUM(Sales),2)  AS annual_sales,
               ROUND(SUM(Profit),2) AS annual_profit
        FROM superstore GROUP BY year ORDER BY year""",

    'Q10 – Top 10 Most Valuable Customers': """
        SELECT "Customer Name", Segment, Region,
               ROUND(SUM(Sales),2)        AS lifetime_sales,
               ROUND(SUM(Profit),2)       AS lifetime_profit,
               COUNT(DISTINCT "Order ID") AS total_orders
        FROM superstore
        GROUP BY "Customer Name", Segment, Region
        ORDER BY lifetime_profit DESC LIMIT 10""",
}

results = {}
for label, sql in biz_questions.items():
    results[label] = run_query(sql, label)


  Q1 – Top 5 Products by Sales


,Product Name,total_sales
0,"Situations Contoured Folding Chairs, 4/Set",2321.05
1,DMI Arturo Collection Mission-style Design Woo...,2204.31
2,Fellowes Officeware Wire Shelving,2155.92
3,Space Solutions HD Industrial Steel Shelving.,2069.46
4,Easy-staple paper,2047.09



  Q2 – Monthly Revenue Trend


,yr_month,monthly_sales
0,0-01-20,1690.63
1,0-02-20,1592.24
2,0-03-20,1280.71
3,0-04-20,1177.77
4,0-05-20,1738.19
...,...,...
302,9-26-20,2488.88
303,9-27-20,371.34
304,9-28-20,1191.72
305,9-29-20,1958.88



  Q3 – Customer Segmentation


...

,Segment,avg_order_value
0,Consumer,117.30
1,Corporate,113.56
2,Home Office,102.83



  Q9 – Year-over-Year Sales


,year,annual_sales,annual_profit
0,0-01,1690.63,272.41
1,0-02,1592.24,401.95
2,0-03,1280.71,120.16
3,0-04,1177.77,190.56
4,0-05,1738.19,254.54
...,...,...,...
302,9-26,2488.88,457.05
303,9-27,371.34,144.74
304,9-28,1191.72,205.72
305,9-29,1958.88,459.86



  Q10 – Top 10 Most Valuable Customers


,Customer Name,Segment,Region,lifetime_sales,lifetime_profit,total_orders
0,Arthur Prichep,Consumer,West,1156.76,271.58,6
1,William Brown,Consumer,West,1333.49,264.53,5
2,Ted Butterfield,Consumer,East,1198.09,256.55,5
3,Rick Wilson,Corporate,West,1240.08,235.82,6
4,Sanjit Chand,Consumer,West,1560.23,223.47,5
5,Brian Thompson,Consumer,East,665.07,214.31,3
6,Arianne Irving,Consumer,West,774.21,210.30,6
7,Emily Phan,Consumer,Central,667.84,190.24,4
8,Denny Blanton,Consumer,East,650.21,185.93,2
9,Ben Peterman,Corporate,West,1082.24,176.63,4


In [16]:
# Cell 16 – Export all results to Excel
os.makedirs('../reports', exist_ok=True)
excel_path = '../reports/task2_sql_results.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    for sheet_name, df_result in results.items():
        df_result.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print(f'All 10 results exported to {excel_path}')

All 10 results exported to ../reports/task2_sql_results.xlsx
